# ComfyUI + Kaggle GPU: reviewed minimal execution notebook
**Manual execution only.** Run preparation in a CPU session first where possible. Enable a GPU only for launch/inference. No external web UI tunnels. Replace `YOUR_USERNAME`. The notebook assumes Internet is enabled for cloning and dependency installs.


## A. CPU preparation — do not allocate GPU yet

In [ ]:
%cd /kaggle/working
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os

os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
import subprocess
from pathlib import Path

APP = Path("/kaggle/working/comfy-kaggle-app")
COMFY = Path("/kaggle/working/ComfyUI")
REPO = "https://github.com/nishanthrjn/comfy-kaggle-app.git"  # EDIT
if not APP.exists():
    subprocess.run(["git", "clone", REPO, str(APP)], check=True)
if not COMFY.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)],
        check=True,
    )
# An existing validated commit may be specified by the app repository.
pin = APP / "config/comfyui-version.txt"
if pin.exists():
    subprocess.run(
        ["git", "-C", str(COMFY), "checkout", "--detach", pin.read_text().strip()],
        check=True,
    )
print(
    "ComfyUI commit:",
    subprocess.check_output(["git", "-C", str(COMFY), "rev-parse", "HEAD"], text=True).strip(),
)

The first successful run should be made with the cloned commit; only **after real image success** record its SHA in `config/comfyui-version.txt` and commit that file to Git. Do not run an early pinning cell that assumes this file already exists.

In [ ]:
!uv venv /kaggle/working/comfy-venv --system-site-packages
!uv pip install --python /kaggle/working/comfy-venv/bin/python -r /kaggle/working/ComfyUI/requirements.txt
!uv sync --project /kaggle/working/comfy-kaggle-app

## B. Checkpoint
Attach the SD1.5 safetensors checkpoint as a Kaggle input **before starting the GPU**. Set `CHECKPOINT_SOURCE` to the exact path visible under `/kaggle/input`; avoid downloading on GPU time. Original model must be legally accessible to your account. If not attached, STOP here.

In [ ]:
from pathlib import Path

CHECKPOINT_SOURCE = Path("/kaggle/input/YOUR_DATASET/v1-5-pruned-emaonly.safetensors")  # EDIT
CHECKPOINT_NAME = "v1-5-pruned-emaonly.safetensors"
assert CHECKPOINT_SOURCE.is_file(), f"Attach checkpoint first: {CHECKPOINT_SOURCE}"
checkpoint_link = COMFY / "models/checkpoints" / CHECKPOINT_NAME
checkpoint_link.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint_link.exists():
    checkpoint_link.symlink_to(CHECKPOINT_SOURCE)
print("Checkpoint available:", checkpoint_link)

## C. GPU-only steps — enable Kaggle GPU accelerator now
If switching accelerators resets your notebook runtime, rerun prerequisite cells as necessary. Confirm GPU type and package compatibility.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(
    [
        "/kaggle/working/comfy-venv/bin/python",
        "-c",
        "import torch; print(torch.__version__,torch.cuda.is_available()); assert torch.cuda.is_available()",
    ],
    check=True,
)

In [ ]:
import subprocess
import time

import requests

with open("/kaggle/working/comfyui.log", "w", encoding="utf-8") as log_handle:
    comfy_process = subprocess.Popen(
        [
            "/kaggle/working/comfy-venv/bin/python",
            "main.py",
            "--listen",
            "127.0.0.1",
            "--port",
            "8188",
        ],
        cwd="/kaggle/working/ComfyUI",
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
print("ComfyUI PID:", comfy_process.pid)
for _ in range(60):
    if comfy_process.poll() is not None:
        raise RuntimeError("ComfyUI exited; inspect /kaggle/working/comfyui.log")
    try:
        r = requests.get("http://127.0.0.1:8188/system_stats", timeout=2)
        if r.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError("ComfyUI did not become ready")
print("ComfyUI ready:", r.json()["system"]["comfyui_version"])

## D. Before inference: verify API graph is accepted
This calls the ComfyUI API's workflow validation endpoint `/object_info` for supported node types but **does not generate an image**. The actual `/prompt` validation occurs during inference.

In [ ]:
import json

workflow = json.loads((APP / "workflows/text_to_image.json").read_text())
node_map = json.loads((APP / "workflows/node_map.json").read_text())
assert workflow and node_map
info = requests.get("http://127.0.0.1:8188/object_info", timeout=30).json()
missing = sorted({n["class_type"] for n in workflow.values()} - set(info))
assert not missing, f"Missing ComfyUI nodes: {missing}"
assert checkpoint_link.exists()
print("Nodes present; ready for one GPU generation")

In [ ]:
import os
import subprocess

# Configure FastAPI to connect to ComfyUI on Kaggle
env = os.environ.copy()

env.update(
    {
        "COMFY_MOCK": "false",
        "COMFY_BASE_URL": "http://127.0.0.1:8188",
        "COMFY_CHECKPOINT": "v1-5-pruned-emaonly.safetensors",
    }
)

# Start FastAPI and capture logs
with open(
    "/kaggle/working/fastapi.log",
    "w",
    encoding="utf-8",
) as api_log:
    app_process = subprocess.Popen(
        [
            "/kaggle/working/comfy-kaggle-app/.venv/bin/python",
            "-m",
            "uvicorn",
            "comfy_kaggle_app.main:app",
            "--host",
            "127.0.0.1",
            "--port",
            "8000",
        ],
        cwd="/kaggle/working/comfy-kaggle-app",
        env=env,
        stdout=api_log,
        stderr=subprocess.STDOUT,
    )

print("FastAPI PID:", app_process.pid)

## E. One real inference test — 512×512 SD1.5 first, not FLUX

In [ ]:
import base64

from IPython.display import Image, display

request_data = {
    "prompt": "cinematic realistic portrait of a baby fox sitting beside an antique lantern in a lush misty forest, detailed fur, golden-hour natural light, professional wildlife photography",
    "negative_prompt": "blurry, watermark, distorted, low quality",
    "seed": 42,
}
r = requests.post("http://127.0.0.1:8000/generate", json=request_data, timeout=480)
print("HTTP:", r.status_code)
if not r.ok:
    print("Response:", r.text[:1000])
r.raise_for_status()
result = r.json()
assert result["status"] == "completed"
image_bytes = base64.b64decode(result["image_base64"], validate=True)
assert image_bytes.startswith(b"\x89PNG\r\n\x1a\n")
output = Path("/kaggle/working/first_comfy_image.png")
output.write_bytes(image_bytes)
print("Saved:", output)
display(Image(data=image_bytes))

## F. Save evidence and stop GPU session
Record exact Git commit, Python/Torch/CUDA versions, selected model SHA256, image and service logs. Download output. **Terminate the Kaggle GPU session using Kaggle's session controls**; terminating only the Python subprocesses does not end notebook GPU allocation.

In [ ]:
print(
    "ComfyUI commit:",
    subprocess.check_output(["git", "-C", str(COMFY), "rev-parse", "HEAD"], text=True).strip(),
)
print("Image:", Path("/kaggle/working/first_comfy_image.png"))
print("Last logs:")
for path in ("/kaggle/working/comfyui.log", "/kaggle/working/fastapi.log"):
    print("\n", path)
    if Path(path).exists():
        print("".join(Path(path).read_text(errors="replace").splitlines(True)[-30:]))